https://github.com/ollama/ollama-python

https://ollama.com/library

https://github.com/ollama/ollama/blob/main/docs/linux.md

In [26]:
# !curl -fsSL https://ollama.com/install.sh | sh

# !pip install ollama

>>> Cleaning up old version at /usr/local/lib/ollama
[sudo] password for mdear2: 


In [1]:
from IPython.display import display, JSON, HTML, clear_output
import json
import subprocess
import time
import ollama

# Verify version
!pip freeze | grep ollama

# Check on GPUs
!nvidia-smi

ollama==0.4.5
ollama-python==0.1.2
Thu Jan 30 08:05:20 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          Off |   0000000C:01:00.0 Off |                    0 |
| N/A   39C    P0             62W /  250W |   22951MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+------------

In [2]:
def unload_ollama_model(model):
    print("\nUnloading Ollama server...")
    data_curl = {
        "model": model,
        "keep_alive": 0
    }
    data_curl_json = json.dumps(data_curl)
    curl_command_unload_LLM = [
        'curl',
        'http://localhost:11434/api/generate',
        '-d',
        data_curl_json
    ]
    result_unload_LLM = subprocess.run(curl_command_unload_LLM, capture_output=True, text=True)
    print("\nUnload LLM:\n", result_unload_LLM.stdout)
    print(result_unload_LLM.stderr)

In [4]:
# List the available models installed on this Ollama server
display(ollama.list().model_dump())

{'models': [{'model': 'deepseek-r1:70b',
   'modified_at': datetime.datetime(2025, 1, 27, 21, 23, 43, 662469, tzinfo=TzInfo(-06:00)),
   'digest': '0c1615a8ca32ef41e433aa420558b4685f9fc7f3fd74119860a8e2e389cd7942',
   'size': 42520397704,
   'details': {'parent_model': '',
    'format': 'gguf',
    'family': 'llama',
    'families': ['llama'],
    'parameter_size': '70.6B',
    'quantization_level': 'Q4_K_M'}},
  {'model': 'qwen2.5-coder:32b',
   'modified_at': datetime.datetime(2025, 1, 9, 22, 8, 33, 824857, tzinfo=TzInfo(-06:00)),
   'digest': '4bd6cbf2d094264457a17aab6bd6acd1ed7a72fb8f8be3cfb193f63c78dd56df',
   'size': 19851349856,
   'details': {'parent_model': '',
    'format': 'gguf',
    'family': 'qwen2',
    'families': ['qwen2'],
    'parameter_size': '32.8B',
    'quantization_level': 'Q4_K_M'}},
  {'model': 'deepseek-coder:33b-instruct-q8_0',
   'modified_at': datetime.datetime(2025, 1, 9, 7, 45, 20, 23398, tzinfo=TzInfo(-06:00)),
   'digest': '1b30ba8b1b95de7e4a65d64b2b37

### Load and Remove LLMs

In [4]:
# Load a new model
llm_model = "deepseek-r1:70b"
ollama.pull(llm_model)

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [3]:
# Delete a model from this server
llm_model = "dolphin3:latest"
ollama.delete(llm_model)

StatusResponse(status='success')

In [13]:
# Unload a model from memory without deleting the LLM
llm_model = "codestral:22b"
unload_ollama_model(llm_model)


Unloading Ollama server...

Unload LLM:
 {"error":"model 'qwen2.5-coder:32b' not found"}
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100    94  100    47  100    47  54272  54272 --:--:-- --:--:-- --:--:-- 94000



# TEST Ollama served LLMs

In [5]:
llm_model = "deepseek-r1:70b"
llm_system_prompt = "You are an expert coder."  # "You are a helpful AI assistant."
llm_user_prompt = "How do you print hello world in Python?"

In [6]:
start_time = time.time()
response = ollama.chat(
    model=llm_model, 
    messages=[
        {"role": "system", "content": llm_system_prompt},
        {'role': 'user', 'content': llm_user_prompt}
    ],
    stream=False,
    options=dict(
        temperature=0.2,
        top_p=0.9,
        num_ctx=128000
    )
)
end_time = time.time() - start_time
print("LLM response time: " + str(round(end_time, 2)) + " seconds.\n\n")
print(response['message']['content'])

LLM response time: 143.41 seconds.


<think>
Okay, so I need to figure out how to print "Hello, World!" in Python. Hmm, where do I start? I remember that Python is a programming language, and printing something is one of the basic functions. But wait, what's the exact syntax for it?

I think there's a function called print() that you use to display output. So maybe I just need to type print("Hello, World!") and run it. But how do I set that up? Do I need to write a script or can I do it in an interactive shell?

If I'm using a Python IDE like PyCharm or VSCode, I probably have to create a new file with a .py extension. Then inside that file, I'll write the print statement. Or maybe I can just open a terminal and type it directly if I'm using the command line.

Wait, what about the quotes? Do they need to be double quotes or single quotes? I think both work in Python as long as they're consistent. So either "Hello, World!" or 'Hello, World!' should be fine.

Let me try writing a simple 

### Test stream LLM interface

In [14]:
llm_model = "deepseek-r1:70b"
llm_system = "You are an expert coder."  # "You are a helpful AI assistant."

In [15]:
llm_user_prompt = "How do you print hello world in Python?"  # "What is your name?"
num_ctx = 128000

In [17]:
start_time = time.time()

# Create a display object to show streaming text
output_display = display(display_id=True)

# Create an HTML display object with a <pre> tag for formatted text
html_display = display(HTML(""), display_id=True)

response_text = ""
stream = ollama.chat(
    model=llm_model,
    messages=[
        {"role": "system", "content": llm_system},
        {'role': 'user', 'content': llm_user_prompt}
    ],
    stream=True,
    options=dict(
        temperature=0.2,
        top_p=0.9,
        num_ctx=num_ctx
    )
)

# Process each chunk in the stream
for chunk in stream:
    # print(chunk['message']['content'], end='', flush=True)
    chunk_content = chunk['message']['content']
    response_text += chunk_content
    # Update the display with current text
    # output_display.update({'text/plain': response_text})
    # Update display with formatted text (preserve newlines and spacing)
    # html_display.update(HTML(
    #    f"<pre style='background:#f8f8f8;padding:10px;border-radius:5px;border:1px solid #ddd;white-space:pre-wrap'>{response_text}</pre>"
    # ))
    # Create typewriter effect with cursor
    display_text = f"{response_text}<span style='border-right:2px solid;animation:blink 1s infinite'> </span>"
    
    html_display.update(HTML(f"""
        <pre style='background:#f8f8f8;padding:10px;border-radius:5px;border:1px solid #ddd;
                      white-space:pre-wrap;font-family:monospace'>
        {display_text}
        </pre>
        <style>
        @keyframes blink {{
            0% {{ opacity:1 }}
            50% {{ opacity:0 }}
            100% {{ opacity:1 }}
        }}
        </style>
    """))

# Clear the streaming display after completion
clear_output(wait=True)
    
end_time = time.time() - start_time
print(response_text)
print("\n\nLLM response time: " + str(round(end_time, 2)) + " seconds.")

<think>
Okay, so I need to figure out how to print "Hello, World!" in Python. Hmm, I've heard that Python is a pretty straightforward language, but I'm not entirely sure where to start. Let me think about what I know.

I remember that in many programming languages, printing something to the screen involves using a function. In Python, I think the function is called print(). So maybe I can use that. But how exactly does it work?

Wait, do I need to put "Hello, World!" in quotes? Probably, because strings are usually enclosed in quotes. So maybe it's like print("Hello, World!"). That seems right.

But wait, what about the syntax? Do I need a semicolon at the end of the line? I don't think so; Python uses line breaks to end statements, so no semicolon is needed. Okay, that makes it simpler.

So putting it all together, I can write a simple script with just one line: print("Hello, World!"). When I run this, it should display "Hello, World!" in the console or wherever the output is directed